# No-Deadline CoMDP+ on Colab (Thesis Prototype)

Minimal demo: same CoMDP+ environment (multi-action, STN, probabilistic outcomes) **without** a deadline; runs the greedy baseline. Prototype for thesis iteration.

In [ ]:
# Clone repo and install deps
%cd /content
!rm -rf tp_mcts
!git clone https://github.com/eliezerRevach/tp_mcts.git
%cd tp_mcts
!pip -q install dill numpy

In [ ]:
# Setup path and argv (unified_planning reads argv at import)
import sys
sys.path.insert(0, "/content/tp_mcts")
sys.argv = [sys.argv[0]]

import random
import numpy as np
import unified_planning as up
from comdp_plus_no_deadline.domains import DOMAIN_FACTORIES
from comdp_plus_no_deadline.engines import MDP, combinationMDP
from comdp_plus_no_deadline.engines.evaluate import evaluation_loop
from comdp_plus_no_deadline.engines.greedy_solver import regular_greedy_plan, combination_greedy_plan
from comdp_plus_no_deadline.scenarios import PRESETS

In [ ]:
def _ground(domain_name, model):
    if domain_name == "nasa_rover":
        grounder = up.engines.compilers.Grounder(model.grounding_map())
    else:
        grounder = up.engines.compilers.Grounder()
    return grounder._compile(model.problem).problem

def build_regular(domain_name, object_amount, garbage_amount=0):
    model = DOMAIN_FACTORIES[domain_name](kind="regular", deadline=None, object_amount=object_amount, garbage_amount=garbage_amount)
    ground = _ground(domain_name, model)
    return up.engines.Convert_problem(ground)._converted_problem

def build_combination(domain_name, object_amount, garbage_amount=0):
    model = DOMAIN_FACTORIES[domain_name](kind="combination", deadline=None, object_amount=object_amount, garbage_amount=garbage_amount)
    ground = _ground(domain_name, model)
    convert = up.engines.Convert_problem_combination(model, ground)
    model.remove_actions(convert._converted_problem)
    return convert._converted_problem

In [ ]:
# Config: pick scenario and mode (edit here)
SCENARIO = "easy_nasa_rover_1"   # easy_stuck_car_1o | easy_nasa_rover_1 | mid_nasa_rover_2 | ...
DOMAIN_TYPE = "regular"           # regular | combination
RUNS = 5
MAX_STEPS = 80
SEED = 123

preset = PRESETS[SCENARIO]
domain_name = preset["domain"]
object_amount = preset["object_amount"]

In [ ]:
random.seed(SEED)
np.random.seed(SEED)

if DOMAIN_TYPE == "regular":
    problem = build_regular(domain_name, object_amount)
    mdp = MDP(problem, discount_factor=0.95, reward_mode="terminal")
    result = evaluation_loop(RUNS, regular_greedy_plan, (mdp, MAX_STEPS, 0.2))
else:
    problem = build_combination(domain_name, object_amount)
    mdp = combinationMDP(problem, discount_factor=0.95, reward_mode="terminal")
    result = evaluation_loop(RUNS, combination_greedy_plan, (mdp, MAX_STEPS, 0.2))

print("=== No-Deadline CoMDP+ Greedy ===")
print(f"scenario={SCENARIO} domain_type={DOMAIN_TYPE}")
print(f"success_rate={result['success_rate']:.3f} avg_makespan={result['avg_makespan']:.3f}")
print(f"avg_plan_length={result['avg_plan_length']:.3f} avg_reward={result['avg_cumulative_reward']:.3f}")